# Research Question 3 - Comparison with Baseline methods 

<!-- All results are saved in `GD4PS/notebooks/rq3_results/[current_time]_results.md` -->

### Methodology 
1. Initialise the dataset. 
2. Initialise the following models: FCNN, GCNN+GAT, GAT, Linegraph Laplacian and SEGNN. 
3. Performance for SE: Train these models on Net 42-A, evaluate the results. 
4. Performance for Generalisability: Train these models on Net 42-B, deploy on Net 42-A, evaluate the results. 
5. Performance for Scalability: Train these models on MVO, evaluate the results. 

In [10]:
import time
import torch 
import os 
import sys 

# to access the models and utils 
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from src.dataset.custom_dataset import NodeEdgeTapDatasetV2 
from src.training.trainer import trainer 
from utils.model_utils import initialize_model, get_eval_results 
from utils.gen_utils import dataset_splitter, get_device, load_config, plot_va_predictions_notebook
from utils.ppnet_utils import initialize_network 
from utils.load_data_utils import load_sampled_input_data 
from src.model.graph_model import FCNNRegressor

# for consistency among all the models 
torch.manual_seed(0) 

## Initialise Dataset

### For Graph Models
This datset contains graph data objects. 

In [11]:
config = load_config()
device = get_device(config['device'])

net = initialize_network(config['data']['net_name'], else_load=config['data']['load_std'], verbose=True)

# config['data']['num_samples'] = 80

start_data_load = time.perf_counter()
sampled_input_data = load_sampled_input_data(sc_type=config['data']['scenario_type'], 
                                                        net=net, 
                                                        num_samples=config['data']['num_samples'],
                                                        noise=config['data']['noise'],
                                                        trafo_ids=[], # foolproof
                                                        scaler=config['data']['scaler'],
                                                        )

end_data_load = time.perf_counter() 
print(f"Dataloading took {end_data_load - start_data_load} seconds.\n\n")

dataset = NodeEdgeTapDatasetV2(model_name=config['model']['name'], sampled_input_data=sampled_input_data)

all_loaders, plot_loader = dataset_splitter(dataset,
                                            batch_size=config['loader']['batch_size'],
                                            split_list=config['loader']['split_list'])

Transformer Indices for DFT_TNP are available from [2,19] 

Number of Trafos = 20 
 
Network: DFT_TNP is selected 

Net DFT_TNP has 42 nodes and 42 edges. 

Scaling inputs...
Number of V, P measurements 18 out of 84

Number of P_to, Q_to, P_from, Q_from measurements 214 out of 252

Dataloading took 0.3294470000000729 seconds.


Dataset for NEGATRegressor selected!


 Directed power flows accounted in dataset...


 get_edge_index_lu handling dictionary of tensors...



In [12]:
dataset[0]

(Data(x=[42, 2], edge_index=[2, 42], y=[42, 2], y_trafo_label=[0], y_tap=[1, 0]),
 Data(x=[42, 6], edge_index=[2, 160], edge_attr=[160], edge_index_u=[2, 0], edge_attr2=[0]),
 Data(x=[42, 6], edge_index=[2, 160], edge_attr=[160]))

### For FCNN model 

This dataset contains topology-agnostic data.
- only considers node input and node label features from `sampled_input_data` 

In [13]:
from src.dataset.custom_dataset import FCNNDataset 
from utils.gen_utils import dataset_splitter_fcnn

fcnn_dataset = FCNNDataset(sampled_input_data=sampled_input_data)

fcnn_all_loaders, fcnn_plot_loader = dataset_splitter_fcnn(fcnn_dataset,
                                            batch_size=config['loader']['batch_size'],
                                            split_list=config['loader']['split_list'])

fcnn_dataset[1] # 42 buses, 2 features = 84 1-D input features
batch = next(iter(fcnn_all_loaders[0]))

## Initialise the models and the respective optimisers

### 1. Topology-Agnostic Fully Connected Neural Network 

In [14]:


num_nodes = len(net.bus.index)
num_node_features = sampled_input_data['node_input_feat'].shape[2]
num_y_node_features = sampled_input_data['y_label'].shape[2]
print(f"Input features = {num_node_features} \n Label features = {num_y_node_features}")

fcnn_model = FCNNRegressor(in_feat = num_node_features * num_nodes, 
                           hid_feat_list=[128], 
                           out_feat=num_y_node_features * num_nodes)

fcnn_total_params = sum(p.numel() for p in fcnn_model.parameters() if p.requires_grad)
print(f'Total number of parameters of model {fcnn_model}: {fcnn_total_params}')

optimizer_fcnn = torch.optim.Adam(fcnn_model.parameters(), 
                                  lr=config['training']['lr'],
                                  weight_decay=0.01,   
                                )
schedular_fcnn = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fcnn,
                                                            mode='min',
                                                            factor=0.1, 
                                                            min_lr=1e-4,
                                                          )

Input features = 2 
 Label features = 2
Total number of parameters of model FCNNRegressor(
  (all_layers): Sequential(
    (0): Linear(in_features=84, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=84, bias=True)
  )
): 21716


### 2. GCNN + GAT model 

In [6]:
model_NGAT = initialize_model(model_name="NGATRegressor",
                            dataset=dataset,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

total_params = sum(p.numel() for p in model_NGAT.parameters() if p.requires_grad)
print(f'Total number of parameters of model {model_NGAT}: {total_params}')


optimizer_NGAT = torch.optim.Adam(model_NGAT.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_NGAT = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_NGAT, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

Total number of parameters of model NGATRegressor(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 4194


### 3. GAT model 

In [7]:
model_GAT = initialize_model(model_name="GATRegressor",
                            dataset=dataset,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

total_params = sum(p.numel() for p in model_GAT.parameters() if p.requires_grad)

print(f'Total number of parameters of model {model_GAT}: {total_params}')

optimizer_GAT = torch.optim.Adam(model_GAT.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_GAT = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_GAT, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

Total number of parameters of model GATRegressor(
  (gatconv): GATConv(2, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 450


### 4. GCNN + Edge-Regression using Linegraph Laplacian + GAT 

In [8]:
model_LGL = initialize_model(model_name="NEGATRegressor_LGL",
                            dataset=dataset,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)

total_params = sum(p.numel() for p in model_LGL.parameters() if p.requires_grad)

print(f'Total number of parameters of model {model_LGL}: {total_params}')

optimizer_LGL = torch.optim.Adam(model_LGL.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_LGL = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_LGL, 
                                                    mode='min',
                                                    factor=0.1,
                                                    min_lr=config['training']['schedular_min_lr'])

Total number of parameters of model NEGATRegressor_LGL(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): TAGConv(6, 128, K=1)
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=64, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 15970


### 5. Proposed SEGNN model 

In [9]:
model = initialize_model(model_name=config['model']['name'],
                            dataset=dataset,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,
                            ).to(device)
                            
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of parameters of model {model}: {total_params}')

optimizer = torch.optim.Adam(model.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

Total number of parameters of model NEGATRegressor(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=64, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 17506


## Performance for State Estimation 

### Training all five models 

In [10]:
# FCNN 
all_losses_FCNN = trainer(model=fcnn_model, 
                            train_loader=fcnn_all_loaders[0],
                            val_loader=fcnn_all_loaders[1],
                            test_loader=fcnn_all_loaders[2],
                            optimizer=optimizer_fcnn, 
                            schedular=schedular_fcnn, 
                            num_epoch=config['training']['num_epochs'],
                            early_stopping=config['training']['early_stopping'],
                            val_patience=config['training']['val_patience'], 
                            device=device,
                            )

At epoch: 0, 	 training loss: 3.183e+00,                     	 validation loss: 1.682e+00 	 lr: 1.00e-02 	 grad_norm: 9.945e+00
At epoch: 1, 	 training loss: 1.653e+00,                     	 validation loss: 9.320e-01 	 lr: 1.00e-02 	 grad_norm: 5.008e+00
At epoch: 2, 	 training loss: 9.247e-01,                     	 validation loss: 9.703e-01 	 lr: 1.00e-02 	 grad_norm: 5.640e-01
At epoch: 3, 	 training loss: 9.649e-01,                     	 validation loss: 9.641e-01 	 lr: 1.00e-02 	 grad_norm: 1.484e-01
At epoch: 4, 	 training loss: 9.576e-01,                     	 validation loss: 9.445e-01 	 lr: 1.00e-02 	 grad_norm: 3.633e-01
At epoch: 5, 	 training loss: 9.381e-01,                     	 validation loss: 9.312e-01 	 lr: 1.00e-02 	 grad_norm: 2.033e-01
At epoch: 6, 	 training loss: 9.244e-01,                     	 validation loss: 8.995e-01 	 lr: 1.00e-02 	 grad_norm: 4.392e-01
At epoch: 7, 	 training loss: 8.934e-01,                     	 validation loss: 8.570e-01 	 lr: 1.00e-02

In [11]:
# GCNN + GAT  
all_losses_NGAT = trainer(model=model_NGAT, 
                    train_loader=all_loaders[0], 
                    val_loader=all_loaders[1], 
                    test_loader=all_loaders[2], 
                    optimizer=optimizer_NGAT,
                    schedular=schedular_NGAT,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

At epoch: 0, 	 training loss: 1.868e+00,                     	 validation loss: 1.277e+00 	 lr: 1.00e-02 	 grad_norm: 4.686e+00
At epoch: 1, 	 training loss: 1.272e+00,                     	 validation loss: 1.353e+00 	 lr: 1.00e-02 	 grad_norm: 3.437e+00
At epoch: 2, 	 training loss: 1.349e+00,                     	 validation loss: 1.092e+00 	 lr: 1.00e-02 	 grad_norm: 3.694e+00
At epoch: 3, 	 training loss: 1.087e+00,                     	 validation loss: 9.144e-01 	 lr: 1.00e-02 	 grad_norm: 2.411e+00
At epoch: 4, 	 training loss: 9.119e-01,                     	 validation loss: 8.898e-01 	 lr: 1.00e-02 	 grad_norm: 1.560e+00
At epoch: 5, 	 training loss: 8.871e-01,                     	 validation loss: 8.071e-01 	 lr: 1.00e-02 	 grad_norm: 3.409e+00
At epoch: 6, 	 training loss: 8.038e-01,                     	 validation loss: 8.228e-01 	 lr: 1.00e-02 	 grad_norm: 2.276e+00
At epoch: 7, 	 training loss: 8.161e-01,                     	 validation loss: 7.162e-01 	 lr: 1.00e-02

In [12]:
# GAT 
all_losses_GAT = trainer(model=model_GAT, 
                    train_loader=all_loaders[0], 
                    val_loader=all_loaders[1], 
                    test_loader=all_loaders[2], 
                    optimizer=optimizer_GAT,
                    schedular=schedular_GAT,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

At epoch: 0, 	 training loss: 2.032e+00,                     	 validation loss: 1.458e+00 	 lr: 1.00e-02 	 grad_norm: 2.639e+00
At epoch: 1, 	 training loss: 1.446e+00,                     	 validation loss: 1.104e+00 	 lr: 1.00e-02 	 grad_norm: 1.820e+00
At epoch: 2, 	 training loss: 1.098e+00,                     	 validation loss: 9.383e-01 	 lr: 1.00e-02 	 grad_norm: 1.141e+00
At epoch: 3, 	 training loss: 9.378e-01,                     	 validation loss: 9.228e-01 	 lr: 1.00e-02 	 grad_norm: 6.461e-01
At epoch: 4, 	 training loss: 9.254e-01,                     	 validation loss: 1.007e+00 	 lr: 1.00e-02 	 grad_norm: 6.980e-01
At epoch: 5, 	 training loss: 1.007e+00,                     	 validation loss: 1.038e+00 	 lr: 1.00e-02 	 grad_norm: 1.266e+00
At epoch: 6, 	 training loss: 1.036e+00,                     	 validation loss: 9.880e-01 	 lr: 1.00e-02 	 grad_norm: 1.459e+00
At epoch: 7, 	 training loss: 9.858e-01,                     	 validation loss: 9.152e-01 	 lr: 1.00e-02

In [13]:
# GCNN + Edge-Regression using Linegraph Laplacian + GAT 
all_losses = trainer(model=model_LGL, 
                    train_loader=all_loaders[0], 
                    val_loader=all_loaders[1], 
                    test_loader=all_loaders[2], 
                    optimizer=optimizer_LGL,
                    schedular=schedular_LGL,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

At epoch: 0, 	 training loss: 1.087e+01,                     	 validation loss: 2.049e+01 	 lr: 1.00e-02 	 grad_norm: 1.065e+02
At epoch: 1, 	 training loss: 2.038e+01,                     	 validation loss: 6.802e+00 	 lr: 1.00e-02 	 grad_norm: 1.633e+02
At epoch: 2, 	 training loss: 6.818e+00,                     	 validation loss: 2.831e+00 	 lr: 1.00e-02 	 grad_norm: 7.176e+01
At epoch: 3, 	 training loss: 2.832e+00,                     	 validation loss: 5.768e+00 	 lr: 1.00e-02 	 grad_norm: 2.406e+01
At epoch: 4, 	 training loss: 5.737e+00,                     	 validation loss: 5.952e+00 	 lr: 1.00e-02 	 grad_norm: 6.100e+01
At epoch: 5, 	 training loss: 5.960e+00,                     	 validation loss: 4.000e+00 	 lr: 1.00e-02 	 grad_norm: 5.991e+01
At epoch: 6, 	 training loss: 4.041e+00,                     	 validation loss: 2.736e+00 	 lr: 1.00e-02 	 grad_norm: 3.870e+01
At epoch: 7, 	 training loss: 2.781e+00,                     	 validation loss: 2.610e+00 	 lr: 1.00e-02

In [14]:
# Proposed 
all_losses = trainer(model=model, 
                    train_loader=all_loaders[0], 
                    val_loader=all_loaders[1], 
                    test_loader=all_loaders[2], 
                    optimizer=optimizer,
                    schedular=schedular,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

At epoch: 0, 	 training loss: 5.436e+00,                     	 validation loss: 5.703e+00 	 lr: 1.00e-02 	 grad_norm: 2.181e+01
At epoch: 1, 	 training loss: 5.573e+00,                     	 validation loss: 3.987e+00 	 lr: 1.00e-02 	 grad_norm: 7.501e+01
At epoch: 2, 	 training loss: 3.992e+00,                     	 validation loss: 1.686e+00 	 lr: 1.00e-02 	 grad_norm: 4.175e+01
At epoch: 3, 	 training loss: 1.679e+00,                     	 validation loss: 2.618e+00 	 lr: 1.00e-02 	 grad_norm: 1.691e+01
At epoch: 4, 	 training loss: 2.588e+00,                     	 validation loss: 2.476e+00 	 lr: 1.00e-02 	 grad_norm: 2.571e+01
At epoch: 5, 	 training loss: 2.461e+00,                     	 validation loss: 2.473e+00 	 lr: 1.00e-02 	 grad_norm: 1.545e+01
At epoch: 6, 	 training loss: 2.464e+00,                     	 validation loss: 1.999e+00 	 lr: 1.00e-02 	 grad_norm: 1.499e+01
At epoch: 7, 	 training loss: 1.991e+00,                     	 validation loss: 1.314e+00 	 lr: 1.00e-02

### Evaluate all five models 

In [15]:
####################### FCNN Model #####################################
results_fcnn = get_eval_results(test_loader=fcnn_all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=fcnn_model, 
                            scaler=sampled_input_data['scaler_y_label'], 
                            fcnn=True, 
                            num_nodes=42)

####################### GCNN+GAT #####################################
results_ngat = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_NGAT, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### GAT #####################################
results_gat = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_GAT, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT_LG #####################################
results_lg = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_LGL, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT Regressor Model #####################################
results = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model, 
                            scaler=sampled_input_data['scaler_y_label'])


RuntimeError: shape '[336, 2]' is invalid for input of size 5120

In [ ]:
def print_results(name, results):
    """Simple function to print results nicely"""
    print(f"\n{'='*50}")
    print(f"{name} Results:")
    print('='*50)
    
    for metric, value in results.items():
        print(f"{metric:25}: {value}")
        print()

In [ ]:
# Usage:
print_results("FCNN", results_fcnn)
print_results("GCNN+GAT", results_ngat)  # Fix: should be results_ngat
print_results("GAT", results_gat)        # Fix: should be results_gat
print_results("LG", results_lg)          # Fix: should be results_lg
print_results("NEGAT", results)          # Fix: should be results



FCNN Results:
Batchwise Average Test Loss: 1.097285e-01

RMSE_V                   : 1.003982e-05

RMSE_A                   : 1.547264e-03

MAE_V                    : 7.276540e-06

MAE_A                    : 6.694188e-04

MaxAE_V                  : 7.094443e-05

MaxAE_A                  : 2.282435e-02

NRMSE_V                  : 9.090758e-07

NRMSE_A                  : 1.646616e-04


GCNN+GAT Results:
Batchwise Average Test Loss: 3.510432e-01

RMSE_V                   : 1.744024e-05

RMSE_A                   : 1.271413e-03

MAE_V                    : 1.446828e-05

MAE_A                    : 9.584049e-04

MaxAE_V                  : 7.334352e-05

MaxAE_A                  : 6.238073e-03

NRMSE_V                  : 1.579160e-06

NRMSE_A                  : 1.351081e-04


GAT Results:
Batchwise Average Test Loss: 4.473648e-01

RMSE_V                   : 2.024801e-05

RMSE_A                   : 1.892623e-03

MAE_V                    : 1.588633e-05

MAE_A                    : 1.212694e-03

Max

## Performance for Generalisability 

### Initialising and training all models for Net-42B

In [ ]:
net_42b = initialize_network(net_name='GHE_NDP2', else_load=config['data']['load_std'], verbose=True)

# config['data']['num_samples'] = 80
config['training']['num_epochs'] = 200

sampled_input_data_42b = load_sampled_input_data(sc_type=config['data']['scenario_type'], 
                                                        net=net_42b, 
                                                        num_samples=config['data']['num_samples'],
                                                        noise=config['data']['noise'],
                                                        trafo_ids=[], # foolproof
                                                        scaler=config['data']['scaler'],
                                                        )

dataset_42b = NodeEdgeTapDatasetV2(model_name=config['model']['name'], sampled_input_data=sampled_input_data_42b)

all_loaders_42b, plot_loader_42b = dataset_splitter(dataset_42b,
                                    batch_size=config['loader']['batch_size'], 
                                    split_list=config['loader']['split_list'])

fcnn_dataset_42b = FCNNDataset(sampled_input_data=sampled_input_data_42b)

fcnn_all_loaders_42b, fcnn_plot_loader_42b = dataset_splitter_fcnn(fcnn_dataset_42b,
                                            batch_size=config['loader']['batch_size'],
                                            split_list=config['loader']['split_list'])

#######################################################################################################
fcnn_model_42b = FCNNRegressor(in_feat = num_node_features * num_nodes, 
                           hid_feat_list=[128], 
                           out_feat=num_y_node_features * num_nodes)

optimizer_fcnn_42b = torch.optim.Adam(fcnn_model_42b.parameters(), 
                                  lr=config['training']['lr'],
                                  weight_decay=0.01,   
                                )
schedular_fcnn_42b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fcnn_42b,
                                                            mode='min',
                                                            factor=0.1, 
                                                            min_lr=1e-4,
                                                          )


#######################################################################################################
model_NGAT_42b = initialize_model(model_name="NGATRegressor",
                            dataset=dataset_42b,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

optimizer_NGAT_42b = torch.optim.Adam(model_NGAT_42b.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_NGAT_42b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_NGAT_42b, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_GAT_42b = initialize_model(model_name="GATRegressor",
                            dataset=dataset_42b,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

optimizer_GAT_42b = torch.optim.Adam(model_GAT_42b.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_GAT_42b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_GAT_42b, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_LGL_42b = initialize_model(model_name="NEGATRegressor_LGL",
                            dataset=dataset_42b,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)

optimizer_LGL_42b = torch.optim.Adam(model_LGL_42b.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_LGL_42b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_LGL_42b, 
                                                    mode='min',
                                                    factor=0.1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_42b = initialize_model(model_name=config['model']['name'],
                            dataset=dataset_42b,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,
                            ).to(device)
                            
optimizer_42b = torch.optim.Adam(model_42b.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_42b = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_42b, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
#######################################################################################################
#######################################################################################################
all_losses_FCNN_42b = trainer(model=fcnn_model_42b, 
                            train_loader=fcnn_all_loaders_42b[0],
                            val_loader=fcnn_all_loaders_42b[1],
                            test_loader=fcnn_all_loaders_42b[2],
                            optimizer=optimizer_fcnn_42b, 
                            schedular=schedular_fcnn_42b, 
                            num_epoch=config['training']['num_epochs'],
                            early_stopping=config['training']['early_stopping'],
                            val_patience=config['training']['val_patience'], 
                            device=device,
                            )

all_losses_NGAT_42b = trainer(model=model_NGAT_42b, 
                    train_loader=all_loaders_42b[0], 
                    val_loader=all_loaders_42b[1], 
                    test_loader=all_loaders_42b[2], 
                    optimizer=optimizer_NGAT_42b,
                    schedular=schedular_NGAT_42b,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_GAT_42b = trainer(model=model_GAT_42b, 
                    train_loader=all_loaders_42b[0], 
                    val_loader=all_loaders_42b[1], 
                    test_loader=all_loaders_42b[2], 
                    optimizer=optimizer_GAT_42b,
                    schedular=schedular_GAT_42b,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_42b = trainer(model=model_LGL_42b, 
                    train_loader=all_loaders_42b[0], 
                    val_loader=all_loaders_42b[1], 
                    test_loader=all_loaders_42b[2], 
                    optimizer=optimizer_LGL_42b,
                    schedular=schedular_LGL_42b,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_42b = trainer(model=model_42b, 
                    train_loader=all_loaders_42b[0], 
                    val_loader=all_loaders_42b[1], 
                    test_loader=all_loaders_42b[2], 
                    optimizer=optimizer_42b,
                    schedular=schedular_42b,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

####################### FCNN Model #####################################
results_fcnn_42b = get_eval_results(test_loader=fcnn_all_loaders_42b[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=fcnn_model_42b, 
                            scaler=sampled_input_data['scaler_y_label'], 
                            fcnn=True, 
                            num_nodes=42)

####################### GCNN+GAT #####################################
results_ngat_42b = get_eval_results(test_loader=all_loaders_42b[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_NGAT_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### GAT #####################################
results_gat_42b = get_eval_results(test_loader=all_loaders_42b[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_GAT_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT_LG #####################################
results_lg_42b = get_eval_results(test_loader=all_loaders_42b[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_LGL_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT Regressor Model #####################################
results_42b = get_eval_results(test_loader=all_loaders_42b[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

Transformer Indices for GHE_NDP2 are available from [2,19] 

Number of Trafos = 20 
 
Network: GHE_NDP2 is selected 

Net GHE_NDP2 has 42 nodes and 40 edges. 

Scaling inputs...
Number of V, P measurements 42 out of 84

Number of P_to, Q_to, P_from, Q_from measurements 219 out of 240

Dataset for NEGATRegressor selected!


 Directed power flows accounted in dataset...


 get_edge_index_lu handling dictionary of tensors...

At epoch: 0, 	 training loss: 3.085e-01,                     	 validation loss: 1.933e-01 	 lr: 1.00e-02 	 grad_norm: 4.591e-01
At epoch: 10, 	 training loss: 1.871e-01,                     	 validation loss: 1.937e-01 	 lr: 1.00e-02 	 grad_norm: 1.435e-01
At epoch: 20, 	 training loss: 1.840e-01,                     	 validation loss: 1.898e-01 	 lr: 1.00e-03 	 grad_norm: 1.116e-01
At epoch: 30, 	 training loss: 1.837e-01,                     	 validation loss: 1.896e-01 	 lr: 1.00e-04 	 grad_norm: 1.071e-01
At epoch: 40, 	 training loss: 1.836e-01,                 

### Training results on Net 42-B

In [ ]:
def print_results(name, results):
    """Simple function to print results nicely"""
    print(f"\n{'='*50}")
    print(f"{name} Results:")
    print('='*50)
    
    for metric, value in results.items():
        print(f"{metric:25}: {value}")
        print()

print_results("FCNN 42-B", results_fcnn_42b)
print_results("GCNN+GAT 42-B", results_ngat_42b)  
print_results("GAT 42-B", results_gat_42b)        
print_results("LG 42-B", results_lg_42b)          
print_results("NEGAT 42-B", results_42b)          


FCNN 42-B Results:
Batchwise Average Test Loss: 1.783730e-01

RMSE_V                   : 1.658401e-05

RMSE_A                   : 1.866024e-03

MAE_V                    : 1.289594e-05

MAE_A                    : 7.710279e-04

MaxAE_V                  : 9.422004e-05

MaxAE_A                  : 1.330104e-02

NRMSE_V                  : 1.501631e-06

NRMSE_A                  : 1.983554e-04


GCNN+GAT 42-B Results:
Batchwise Average Test Loss: 1.099565e-01

RMSE_V                   : 1.329804e-05

RMSE_A                   : 4.519377e-04

MAE_V                    : 9.721899e-06

MAE_A                    : 3.206723e-04

MaxAE_V                  : 1.064837e-04

MaxAE_A                  : 2.645478e-03

NRMSE_V                  : 1.204098e-06

NRMSE_A                  : 4.805093e-05


GAT 42-B Results:
Batchwise Average Test Loss: 4.686342e-01

RMSE_V                   : 2.316986e-05

RMSE_A                   : 2.831749e-03

MAE_V                    : 1.789190e-05

MAE_A                    : 1.

### Deployment results on Net 42-A

In [ ]:
####################### FCNN Model #####################################
results_fcnn_42a_gen = get_eval_results(test_loader=fcnn_all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=fcnn_model_42b, 
                            scaler=sampled_input_data['scaler_y_label'], 
                            fcnn=True, 
                            num_nodes=42)

####################### GCNN+GAT #####################################
results_ngat_42a_gen = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_NGAT_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### GAT #####################################
results_gat_42a_gen = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_GAT_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT_LG #####################################
results_lg_42a_gen = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_LGL_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT Regressor Model #####################################
results_42a_gen = get_eval_results(test_loader=all_loaders[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_42b, 
                            scaler=sampled_input_data['scaler_y_label'])

print_results("FCNN _42a_gen", results_fcnn_42a_gen)
print_results("GCNN+GAT _42a_gen", results_ngat_42a_gen)  
print_results("GAT _42a_gen", results_gat_42a_gen)        
print_results("LG _42a_gen", results_lg_42a_gen)          
print_results("NEGAT _42a_gen", results_42a_gen)       

Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.

FCNN _42a_gen Results:
Batchwise Average Test Loss: 7.014629e-01

RMSE_V                   : 4.196156e-05

RMSE_A                   : 3.569374e-03

MAE_V                    : 3.756566e-05

MAE_A                    : 2.506717e-03

MaxAE_V                  : 1.545697e-04

MaxAE_A                  : 2.797367e-02

NRMSE_V                  : 3.799496e-06

NRMSE_A                  : 3.798568e-04


GCNN+GAT _42a_gen Results:
Batchwise Average Test Loss: 6.610510e-01

RMSE_V                   : 3.130082e-05

RMSE_A                   : 2.433884e-03

MAE_V                    : 2.061711e-05

MAE_A                    : 1.852042e-03

MaxAE_V                  : 1.063049e-04

MaxAE_A                  : 9.633884e-03

NRMSE_V                  : 2.834194e-06

NRMSE_A    

### Performane for Scalability 

In [ ]:
config['data']['load_std'] = 0.3
net_MVO = initialize_network(net_name='MVO', else_load=config['data']['load_std'], verbose=True)

# config['data']['num_samples'] = 80

sampled_input_data_MVO = load_sampled_input_data(sc_type=config['data']['scenario_type'], 
                                                        net=net_MVO, 
                                                        num_samples=config['data']['num_samples'],
                                                        noise=config['data']['noise'],
                                                        trafo_ids=[], # foolproof
                                                        scaler=config['data']['scaler'],
                                                        )

dataset_MVO = NodeEdgeTapDatasetV2(model_name=config['model']['name'], sampled_input_data=sampled_input_data_MVO)

all_loaders_MVO, plot_loader_MVO = dataset_splitter(dataset_MVO,
                                    batch_size=config['loader']['batch_size'], 
                                    split_list=config['loader']['split_list'])

fcnn_dataset_MVO = FCNNDataset(sampled_input_data=sampled_input_data_MVO)

fcnn_all_loaders_MVO, fcnn_plot_loader_MVO = dataset_splitter_fcnn(fcnn_dataset_MVO,
                                            batch_size=config['loader']['batch_size'],
                                            split_list=config['loader']['split_list'])

num_nodes = len(net_MVO.bus.index)

#######################################################################################################
fcnn_model_MVO = FCNNRegressor(in_feat = num_node_features * num_nodes, 
                           hid_feat_list=[128], 
                           out_feat=num_y_node_features * num_nodes)

optimizer_fcnn_MVO = torch.optim.Adam(fcnn_model_MVO.parameters(), 
                                  lr=config['training']['lr'],
                                  weight_decay=0.01,   
                                )
schedular_fcnn_MVO = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fcnn_MVO,
                                                            mode='min',
                                                            factor=0.1, 
                                                            min_lr=1e-4,
                                                          )


#######################################################################################################
model_NGAT_MVO = initialize_model(model_name="NGATRegressor",
                            dataset=dataset_MVO,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

optimizer_NGAT_MVO = torch.optim.Adam(model_NGAT_MVO.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_NGAT_MVO = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_NGAT_MVO, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_GAT_MVO = initialize_model(model_name="GATRegressor",
                            dataset=dataset_MVO,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)    

optimizer_GAT_MVO = torch.optim.Adam(model_GAT_MVO.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_GAT_MVO = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_GAT_MVO, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_LGL_MVO = initialize_model(model_name="NEGATRegressor_LGL",
                            dataset=dataset_MVO,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,  
                            ).to(device)

optimizer_LGL_MVO = torch.optim.Adam(model_LGL_MVO.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_LGL_MVO = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_LGL_MVO, 
                                                    mode='min',
                                                    factor=0.1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
model_MVO = initialize_model(model_name=config['model']['name'],
                            dataset=dataset_MVO,
                            node_out_features=config['model']['node_out_features'],
                            list_node_hidden_features=config['model']['list_node_hidden_features'],
                            k_hop_node=config['model']['k_hop_node'],
                            edge_out_features=config['model']['edge_out_features'], 
                            list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                            k_hop_edge=config['model']['k_hop_edge'],
                            trafo_hop=config['model']['trafo_hop'],
                            edge_index_list=sampled_input_data['edge_index'],
                            gat_out_features=config['model']['gat_out_features'],
                            gat_head=config['model']['gat_head'],
                            bias=config['model']['bias'], 
                            normalize=config['model']['normalize'], 
                            device=device,
                            ).to(device)
                            
optimizer_MVO = torch.optim.Adam(model_MVO.parameters(),lr=config['training']['lr'], weight_decay=config['training']['weight_decay'])

schedular_MVO = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer_MVO, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    # patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

#######################################################################################################
#######################################################################################################
#######################################################################################################
all_losses_FCNN_MVO = trainer(model=fcnn_model_MVO, 
                            train_loader=fcnn_all_loaders_MVO[0],
                            val_loader=fcnn_all_loaders_MVO[1],
                            test_loader=fcnn_all_loaders_MVO[2],
                            optimizer=optimizer_fcnn_MVO, 
                            schedular=schedular_fcnn_MVO, 
                            num_epoch=config['training']['num_epochs'],
                            early_stopping=config['training']['early_stopping'],
                            val_patience=config['training']['val_patience'], 
                            device=device,
                            )

all_losses_NGAT_MVO = trainer(model=model_NGAT_MVO, 
                    train_loader=all_loaders_MVO[0], 
                    val_loader=all_loaders_MVO[1], 
                    test_loader=all_loaders_MVO[2], 
                    optimizer=optimizer_NGAT_MVO,
                    schedular=schedular_NGAT_MVO,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_GAT_MVO = trainer(model=model_GAT_MVO, 
                    train_loader=all_loaders_MVO[0], 
                    val_loader=all_loaders_MVO[1], 
                    test_loader=all_loaders_MVO[2], 
                    optimizer=optimizer_GAT_MVO,
                    schedular=schedular_GAT_MVO,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_MVO = trainer(model=model_LGL_MVO, 
                    train_loader=all_loaders_MVO[0], 
                    val_loader=all_loaders_MVO[1], 
                    test_loader=all_loaders_MVO[2], 
                    optimizer=optimizer_LGL_MVO,
                    schedular=schedular_LGL_MVO,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)

all_losses_MVO = trainer(model=model_MVO, 
                    train_loader=all_loaders_MVO[0], 
                    val_loader=all_loaders_MVO[1], 
                    test_loader=all_loaders_MVO[2], 
                    optimizer=optimizer_MVO,
                    schedular=schedular_MVO,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'],  
                    device=device)


Transformer Indices for MVO are available from [0,141] 

Number of Trafos = 143 
 
Network: MVO is selected 

Net MVO has 320 nodes and 318 edges. 

Filling nan as 0 in tap_pos, tap_neutral, tap_step_degree
Scaling inputs...
Number of V, P measurements 310 out of 640

Number of P_to, Q_to, P_from, Q_from measurements 1777 out of 1944

Dataset for NEGATRegressor selected!


 Directed power flows accounted in dataset...


 get_edge_index_lu handling dictionary of tensors...

At epoch: 0, 	 training loss: 3.977e-01,                     	 validation loss: 7.404e-02 	 lr: 1.00e-02 	 grad_norm: 5.905e-01
At epoch: 10, 	 training loss: 7.293e-02,                     	 validation loss: 7.357e-02 	 lr: 1.00e-02 	 grad_norm: 1.138e-01
At epoch: 20, 	 training loss: 7.258e-02,                     	 validation loss: 7.379e-02 	 lr: 1.00e-02 	 grad_norm: 1.084e-01
At epoch: 30, 	 training loss: 7.208e-02,                     	 validation loss: 7.508e-02 	 lr: 1.00e-02 	 grad_norm: 1.013e-01
At epoc

### Scalability results

In [ ]:
####################### FCNN Model #####################################
results_fcnn_MVO = get_eval_results(test_loader=fcnn_all_loaders_MVO[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=fcnn_model_MVO, 
                            scaler=sampled_input_data['scaler_y_label'], 
                            fcnn=True, 
                            num_nodes=320)

####################### GCNN+GAT #####################################
results_ngat_MVO = get_eval_results(test_loader=all_loaders_MVO[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_NGAT_MVO, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### GAT #####################################
results_gat_MVO = get_eval_results(test_loader=all_loaders_MVO[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_GAT_MVO, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT_LG #####################################
results_lg_MVO = get_eval_results(test_loader=all_loaders_MVO[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_LGL_MVO, 
                            scaler=sampled_input_data['scaler_y_label'])

####################### NEGAT Regressor Model #####################################
results_MVO = get_eval_results(test_loader=all_loaders_MVO[2],
                            tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model_MVO, 
                            scaler=sampled_input_data['scaler_y_label'])

Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.
Calculating results for StandardScaled Voltage and Angles.


In [ ]:
print_results("FCNN MVO", results_fcnn_MVO)
print_results("GCNN+GAT MVO", results_ngat_MVO)  
print_results("GAT MVO", results_gat_MVO)        
print_results("LG MVO", results_lg_MVO)          
print_results("NEGAT MVO", results_MVO)


FCNN MVO Results:
Batchwise Average Test Loss: 7.031029e-02

RMSE_V                   : 1.604969e-04

RMSE_A                   : 7.339825e-05

MAE_V                    : 9.370753e-05

MAE_A                    : 5.555942e-05

MaxAE_V                  : 1.413375e-03

MaxAE_A                  : 3.942102e-04

NRMSE_V                  : 1.453125e-05

NRMSE_A                  : 7.801833e-06


GCNN+GAT MVO Results:
Batchwise Average Test Loss: 8.996694e-02

RMSE_V                   : 3.689402e-05

RMSE_A                   : 9.687555e-04

MAE_V                    : 2.739824e-05

MAE_A                    : 2.392767e-04

MaxAE_V                  : 5.804002e-04

MaxAE_A                  : 7.925615e-03

NRMSE_V                  : 3.340384e-06

NRMSE_A                  : 1.029737e-04


GAT MVO Results:
Batchwise Average Test Loss: 7.277751e-01

RMSE_V                   : 3.687712e-05

RMSE_A                   : 2.592603e-03

MAE_V                    : 2.592432e-05

MAE_A                    : 2.154